# MCP Server

Starts the FastMCP server that provides agent card discovery (via embeddings), travel data queries, and Google Places lookup.

In [1]:
import json
import logging
import os
import sqlite3
import sys
from pathlib import Path

import numpy as np
import requests
import weave
from openai import OpenAI
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger(__name__)

WANDB_PROJECT = os.getenv('WANDB_WORKSPACE')
weave_client = weave.init(WANDB_PROJECT)
print(f'Weave initialized: {WANDB_PROJECT}')

AGENT_CARDS_DIR = 'agent_cards'
EMBEDDING_MODEL = 'text-embedding-3-small'
SQLITE_DB = 'travel_agency.db'
PLACES_API_URL = 'https://places.googleapis.com/v1/places:searchText'
HOST, PORT = 'localhost', 10100

oai = OpenAI()

/Users/paul/.cache/uv/archive-v0/pqxv7ELpHFQOMM0CmHnVk/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


2026-04-19 15:25:23,697 INFO httpx: HTTP Request: GET https://trace.wandb.ai/server_info "HTTP/1.1 200 OK"
2026-04-19 15:25:24,769 INFO httpx: HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
2026-04-19 15:25:25,111 INFO httpx: HTTP Request: POST https://api.wandb.ai/graphql "HTTP/1.1 200 OK"
2026-04-19 15:25:25,192 INFO httpx: HTTP Request: GET https://trace.wandb.ai/server_info "HTTP/1.1 200 OK"
2026-04-19 15:25:25,294 INFO httpx: HTTP Request: GET https://pypi.org/pypi/wandb/json "HTTP/1.1 200 OK"
2026-04-19 15:25:25,434 INFO httpx: HTTP Request: GET https://pypi.org/pypi/weave/json "HTTP/1.1 200 OK"


weave: weave version 0.52.37 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade


2026-04-19 15:25:25,452 INFO weave.trace.init_message: weave version 0.52.37 is available!  To upgrade, please run:
 $ pip install weave --upgrade


weave: Logged in as Weights & Biases user: paulbruffett.
weave: View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave


2026-04-19 15:25:25,454 INFO weave.trace.init_message: Logged in as Weights & Biases user: paulbruffett.
View Weave data at https://wandb.ai/pbruffett/a2a-lab/weave
Weave initialized: pbruffett/a2a-lab


In [2]:
def load_agent_cards():
    card_uris, agent_cards = [], []
    for filename in sorted(os.listdir(AGENT_CARDS_DIR)):
        if filename.endswith('.json'):
            with open(Path(AGENT_CARDS_DIR) / filename) as f:
                data = json.load(f)
            card_uris.append(f'resource://agent_cards/{Path(filename).stem}')
            agent_cards.append(data)
    return card_uris, agent_cards


def build_agent_card_embeddings():
    card_uris, agent_cards = load_agent_cards()
    texts = [json.dumps(card) for card in agent_cards]
    result = oai.embeddings.create(input=texts, model=EMBEDDING_MODEL)
    embeddings = [item.embedding for item in result.data]
    print(f'Loaded {len(card_uris)} agent cards with embeddings')
    return card_uris, agent_cards, np.array(embeddings)


card_uris, agent_cards, card_embeddings = build_agent_card_embeddings()

weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019da7d9-3e74-7d22-b6ff-6f03c3764f80


2026-04-19 15:25:25,625 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019da7d9-3e74-7d22-b6ff-6f03c3764f80
2026-04-19 15:25:26,779 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Loaded 5 agent cards with embeddings


In [3]:
mcp = FastMCP('agent-cards', host=HOST, port=PORT)


@mcp.tool(name='find_agent', description='Find the most relevant agent card for a query.')
@weave.op()
def find_agent(query: str) -> str:
    logger.info('[mcp.find_agent] query=%r', query)
    result = oai.embeddings.create(input=[query], model=EMBEDDING_MODEL)
    query_embedding = result.data[0].embedding
    scores = card_embeddings @ query_embedding
    chosen = agent_cards[int(np.argmax(scores))]
    logger.info('[mcp.find_agent] chosen=%s', chosen.get('name'))
    return json.dumps(chosen)


@mcp.tool()
@weave.op()
def query_places_data(query: str):
    """Query Google Places."""
    logger.info('[mcp.query_places_data] query=%r', query)
    api_key = os.getenv('GOOGLE_PLACES_API_KEY')
    if not api_key:
        return {'places': []}
    headers = {
        'X-Goog-Api-Key': api_key,
        'X-Goog-FieldMask': 'places.id,places.displayName,places.formattedAddress',
        'Content-Type': 'application/json',
    }
    try:
        resp = requests.post(PLACES_API_URL, headers=headers, json={'textQuery': query, 'languageCode': 'en', 'maxResultCount': 10})
        resp.raise_for_status()
        data = resp.json()
        logger.info('[mcp.query_places_data] places=%d', len(data.get('places', [])))
        return data
    except Exception as e:
        logger.warning('[mcp.query_places_data] error=%s', e)
        return {'places': []}


@mcp.tool()
@weave.op()
def query_travel_data(query: str) -> dict:
    """Query the travel SQLite database (flights, hotels, rental_cars). Only SELECT queries."""
    logger.info('[mcp.query_travel_data] query=%r', query)
    if not query or not query.strip().upper().startswith('SELECT'):
        raise ValueError(f'Invalid query: {query}')
    try:
        with sqlite3.connect(SQLITE_DB) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.cursor().execute(query).fetchall()
            result = json.dumps({'results': [dict(row) for row in rows]})
            logger.info('[mcp.query_travel_data] rows=%d', len(rows))
            return result
    except Exception as e:
        logger.warning('[mcp.query_travel_data] error=%s', e)
        return {'error': str(e)}


@mcp.resource('resource://agent_cards/list', mime_type='application/json')
def get_agent_cards() -> dict:
    return {'agent_cards': list(card_uris)}


@mcp.resource('resource://agent_cards/{card_name}', mime_type='application/json')
def get_agent_card(card_name: str) -> dict:
    uri = f'resource://agent_cards/{card_name}'
    matches = [card for u, card in zip(card_uris, agent_cards) if u == uri]
    return {'agent_card': matches}

In [ ]:
print(f'MCP Server running at http://{HOST}:{PORT}')
await mcp.run_sse_async()

MCP Server running at http://localhost:10100


INFO:     Started server process [22897]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:10100 (Press CTRL+C to quit)


INFO:     ::1:56707 - "GET /sse HTTP/1.1" 200 OK
2026-04-19 15:25:27,564 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
2026-04-19 15:25:29,612 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
INFO:     ::1:56714 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:56715 - "POST /messages/?session_id=5aa5d1f53e7c4507afd0d260160586e4 HTTP/1.1" 202 Accepted
INFO:     ::1:56715 - "POST /messages/?session_id=5aa5d1f53e7c4507afd0d260160586e4 HTTP/1.1" 202 Accepted
INFO:     ::1:56715 - "POST /messages/?session_id=5aa5d1f53e7c4507afd0d260160586e4 HTTP/1.1" 202 Accepted
2026-04-19 15:25:33,931 INFO mcp.server.lowlevel.server: Processing request of type ReadResourceRequest


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019da7d9-5eeb-7fda-8df8-3f606f4037b2


2026-04-19 15:25:33,939 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019da7d9-5eeb-7fda-8df8-3f606f4037b2
2026-04-19 15:25:35,566 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
INFO:     ::1:56839 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:56840 - "POST /messages/?session_id=a2d3963df0e447a1b11a804fcd7fd302 HTTP/1.1" 202 Accepted
INFO:     ::1:56840 - "POST /messages/?session_id=a2d3963df0e447a1b11a804fcd7fd302 HTTP/1.1" 202 Accepted
INFO:     ::1:56840 - "POST /messages/?session_id=a2d3963df0e447a1b11a804fcd7fd302 HTTP/1.1" 202 Accepted
2026-04-19 15:28:04,406 INFO mcp.server.lowlevel.server: Processing request of type ReadResourceRequest


...

2026-04-20 13:13:49,129 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac87-1cc8-7b03-9137-a2eaf808aaf8
2026-04-20 13:13:49,891 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
2026-04-20 13:13:50,293 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-20 13:13:50,300 INFO __main__: [mcp.find_agent] chosen=Hotel Booking Agent
INFO:     ::1:61446 - "POST /messages/?session_id=17f2d33ad8ce4c828770030d4faec034 HTTP/1.1" 202 Accepted
2026-04-20 13:13:50,305 INFO mcp.server.lowlevel.server: Processing request of type ListToolsRequest
INFO:     ::1:61448 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:61449 - "POST /messages/?session_id=a6b8a7ad831341a58899f15e777eee1a HTTP/1.1" 202 Accepted
INFO:     ::1:61449 - "POST /messages/?session_id=a6b8a7ad831341a58899f15e777eee1a HTTP/1.1" 202 Accepted
INFO:     ::1:61449 - "POST /messages/?session_id=a6b8a7ad831341a58899f15e777eee1a HTTP/1

weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac87-2acd-7182-83d3-4e300d3bcf14


2026-04-20 13:13:52,718 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac87-2acd-7182-83d3-4e300d3bcf14
2026-04-20 13:13:52,720 INFO __main__: [mcp.query_travel_data] rows=1
2026-04-20 13:13:54,125 INFO httpx: HTTP Request: POST https://trace.wandb.ai/call/upsert_batch "HTTP/1.1 200 OK"
INFO:     ::1:61450 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:61451 - "POST /messages/?session_id=c693f47186024e18bf969e2ebb81c00d HTTP/1.1" 202 Accepted
INFO:     ::1:61451 - "POST /messages/?session_id=c693f47186024e18bf969e2ebb81c00d HTTP/1.1" 202 Accepted
INFO:     ::1:61451 - "POST /messages/?session_id=c693f47186024e18bf969e2ebb81c00d HTTP/1.1" 202 Accepted
2026-04-20 13:13:55,292 INFO mcp.server.lowlevel.server: Processing request of type CallToolRequest
2026-04-20 13:13:55,292 INFO __main__: [mcp.find_agent] query='No car rental required for this trip.'


weave: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac87-34dc-7dee-bacc-e545f7364acc


2026-04-20 13:13:55,293 INFO weave.trace.weave_client: 🍩 https://wandb.ai/pbruffett/a2a-lab/r/call/019dac87-34dc-7dee-bacc-e545f7364acc
2026-04-20 13:13:56,152 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-04-20 13:13:56,159 INFO __main__: [mcp.find_agent] chosen=Car Rental Agent
INFO:     ::1:61451 - "POST /messages/?session_id=c693f47186024e18bf969e2ebb81c00d HTTP/1.1" 202 Accepted
2026-04-20 13:13:56,165 INFO mcp.server.lowlevel.server: Processing request of type ListToolsRequest
INFO:     ::1:61453 - "GET /sse HTTP/1.1" 200 OK
INFO:     ::1:61454 - "POST /messages/?session_id=5c5b223d29c443fab2870f2928ef4703 HTTP/1.1" 202 Accepted
INFO:     ::1:61454 - "POST /messages/?session_id=5c5b223d29c443fab2870f2928ef4703 HTTP/1.1" 202 Accepted
INFO:     ::1:61454 - "POST /messages/?session_id=5c5b223d29c443fab2870f2928ef4703 HTTP/1.1" 202 Accepted
2026-04-20 13:13:56,220 INFO mcp.server.lowlevel.server: Processing request of type ListToolsRequest